# Đọc dữ liệu
- Nhiệm vụ chính là đọc dữ liệu từ file csv sang parquet bằng Spark.
- Vì dữ liệu ban đầu rất lớn, mục tiêu của chúng ta là sẽ chỉ lấy ra những phần quan trong, lọc và loại bỏ những cột không quan trọng.

### Buoc 1: Khoi tao SparkSession

Muc tieu cua buoc nay:
- Tao SparkSession de lam viec voi du lieu lon bang PySpark.
- Chay Spark o che do local tren may ca nhan.
- Khai bao san cac duong dan input/output cho cac buoc tiep theo.
- Kiem tra nhanh Spark version va duong dan file CSV.

#### Viec can lam trong buoc 1

1. Kiem tra Python environment da co pyspark hay chua.
2. Kiem tra may da cai Java va Java version co phu hop voi Spark hay chua.
3. Tao SparkSession bang SparkSession.builder.
4. Khai bao cac duong dan input/output dung cho cac buoc tiep theo.
5. In thong tin Spark de xac nhan khoi tao thanh cong.

In [1]:
import importlib.util
import re
import shutil
import subprocess

SUPPORTED_JAVA_MAJOR_VERSIONS = {17, 21}

if importlib.util.find_spec("pyspark") is None:
    raise ModuleNotFoundError(
        "Chua cai pyspark trong kernel hien tai. "
        "Hay chay: python -m pip install pyspark"
    )

java_path = shutil.which("java")
if java_path is None:
    raise RuntimeError(
        "Chua tim thay Java trong PATH. "
        "Hay cai JDK va mo lai VS Code/Jupyter de PATH duoc cap nhat."
    )

java_check = subprocess.run(
    [java_path, "-version"],
    capture_output=True,
    text=True,
    check=False,
)
java_version_text = java_check.stderr or java_check.stdout
first_java_line = java_version_text.splitlines()[0] if java_version_text else "unknown"
java_version_match = re.search(r'\"(\d+)', java_version_text)

if java_version_match is None:
    raise RuntimeError(f"Khong doc duoc Java version tu: {first_java_line}")

java_major_version = int(java_version_match.group(1))
if java_major_version not in SUPPORTED_JAVA_MAJOR_VERSIONS:
    raise RuntimeError(
        f"Java hien tai la version {java_major_version}, chua phu hop voi Spark. "
        "Hay dung JDK 17 hoac JDK 21, sau do cap nhat JAVA_HOME/PATH va restart VS Code/Jupyter."
    )

print("OK: pyspark da san sang")
print("OK: Java version phu hop voi Spark")
print("Java path:", java_path)
print("Java version:", first_java_line)


OK: pyspark da san sang
OK: Java version phu hop voi Spark
Java path: C:\Program Files\Common Files\Oracle\Java\javapath\java.EXE
Java version: java version "21.0.11" 2026-04-21 LTS


In [2]:
import os
import sys
from pathlib import Path

from pyspark.sql import SparkSession

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ["PATH"]


# Dam bao Spark driver va Python worker dung cung mot Python executable.
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Notebook co the duoc chay tu thu muc goc project hoac tu week_3.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "week_3" else Path.cwd()

DATA_DIR = PROJECT_ROOT / "data_pyspark"
PARQUET_DIR = PROJECT_ROOT / "data_pyspark_parquet"

TRAIN_CSV_PATH = DATA_DIR / "train_v2.csv"
TEST_CSV_PATH = DATA_DIR / "test_v2.csv"

TRAIN_PARQUET_PATH = PARQUET_DIR / "train_sessions"
TEST_PARQUET_PATH = PARQUET_DIR / "test_sessions"
LOG_PATH = PARQUET_DIR / "read_csv_to_parquet_log.json"

spark = (
    SparkSession.builder
    .appName("GA Customer Revenue - CSV to Parquet") # 
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .config("spark.pyspark.python", sys.executable)
    .config("spark.pyspark.driver.python", sys.executable)
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)
print("Python executable:", sys.executable)
print("Project root:", PROJECT_ROOT)
print("Train CSV exists:", TRAIN_CSV_PATH.exists())
print("Test CSV exists:", TEST_CSV_PATH.exists())


Spark version: 4.1.2
Spark master: local[*]
Python executable: f:\ide\anaconda\python.exe
Project root: g:\ds
Train CSV exists: True
Test CSV exists: True


### Buoc 2: Doc thu header va schema CSV

Viec can lam trong buoc 2:
1. Khai bao option doc CSV phu hop voi file co cot JSON dang string.
2. Doc header cua train/test bang Spark ma khong scan toan bo file.
3. In so luong cot va danh sach cot.
4. In schema mac dinh cua Spark khi chua parse/cast kieu du lieu.
5. So sanh cot train va test de biet co cot nao chi xuat hien o mot file hay khong.

In [3]:
CSV_READ_OPTIONS = {
    "header": "true",
    "quote": '"',
    "escape": '"',
    "multiLine": "false",
    "mode": "PERMISSIVE",
}

print(f"Train CSV size: {TRAIN_CSV_PATH.stat().st_size / 1024**3:.2f} GB")
print(f"Test CSV size: {TEST_CSV_PATH.stat().st_size / 1024**3:.2f} GB")

train_header_df = (
    spark.read
    .options(**CSV_READ_OPTIONS)
    .csv(str(TRAIN_CSV_PATH))
    .limit(0)
)

test_header_df = (
    spark.read
    .options(**CSV_READ_OPTIONS)
    .csv(str(TEST_CSV_PATH))
    .limit(0)
)


def show_header_and_schema(name, df):
    print(f"\n{name} column count:", len(df.columns))
    print(f"{name} columns:")
    for index, column_name in enumerate(df.columns, start=1):
        print(f"{index:02d}. {column_name}")
    print(f"\n{name} schema:")
    df.printSchema()


show_header_and_schema("train_v2", train_header_df)
show_header_and_schema("test_v2", test_header_df)

train_only_columns = sorted(set(train_header_df.columns) - set(test_header_df.columns))
test_only_columns = sorted(set(test_header_df.columns) - set(train_header_df.columns))

print("\nColumns only in train:", train_only_columns)
print("Columns only in test:", test_only_columns)


Train CSV size: 23.67 GB
Test CSV size: 7.09 GB

train_v2 column count: 13
train_v2 columns:
01. channelGrouping
02. customDimensions
03. date
04. device
05. fullVisitorId
06. geoNetwork
07. hits
08. socialEngagementType
09. totals
10. trafficSource
11. visitId
12. visitNumber
13. visitStartTime

train_v2 schema:
root
 |-- channelGrouping: string (nullable = true)
 |-- customDimensions: string (nullable = true)
 |-- date: string (nullable = true)
 |-- device: string (nullable = true)
 |-- fullVisitorId: string (nullable = true)
 |-- geoNetwork: string (nullable = true)
 |-- hits: string (nullable = true)
 |-- socialEngagementType: string (nullable = true)
 |-- totals: string (nullable = true)
 |-- trafficSource: string (nullable = true)
 |-- visitId: string (nullable = true)
 |-- visitNumber: string (nullable = true)
 |-- visitStartTime: string (nullable = true)


test_v2 column count: 13
test_v2 columns:
01. channelGrouping
02. customDimensions
03. date
04. device
05. fullVisitorId
06

### Buoc 3: Doc sample nho

Viec can lam trong buoc 3:
1. Doc mot so dong nho tu train/test de inspect du lieu that.
2. Cache sample tam thoi vi ta se xem nhieu lan trong notebook.
3. Kiem tra so dong sample doc duoc.
4. Xem nhanh cac cot session-level de hieu grain cua du lieu.
5. Xem preview cac cot JSON dang string de chuan bi cho buoc parse JSON tiep theo.

In [4]:
from pyspark.sql import functions as F


SAMPLE_ROW_COUNT = 100
PREVIEW_ROW_COUNT = 5
PREVIEW_TEXT_LENGTH = 180


def read_csv_sample(csv_path, row_count=SAMPLE_ROW_COUNT):
    full_df = spark.read.options(**CSV_READ_OPTIONS).csv(str(csv_path))
    sample_rows = full_df.take(row_count)
    sample_df = spark.createDataFrame(sample_rows, schema=full_df.schema).cache()
    return sample_df


train_sample_df = read_csv_sample(TRAIN_CSV_PATH)
test_sample_df = read_csv_sample(TEST_CSV_PATH)

train_sample_count = train_sample_df.count()
test_sample_count = test_sample_df.count()

print("Train sample rows:", train_sample_count)
print("Test sample rows:", test_sample_count)

session_preview_columns = [
    "date",
    "fullVisitorId",
    "visitId",
    "visitNumber",
    "visitStartTime",
    "channelGrouping",
    "socialEngagementType",
]

print("\nTrain session-level preview:")
train_sample_df.select(*session_preview_columns).show(PREVIEW_ROW_COUNT, truncate=80)

print("\nTest session-level preview:")
test_sample_df.select(*session_preview_columns).show(PREVIEW_ROW_COUNT, truncate=80)

json_columns = ["customDimensions", "device", "geoNetwork", "hits", "totals", "trafficSource"]


def show_json_preview(name, df):
    preview_expressions = [
        F.substring(F.col(column_name), 1, PREVIEW_TEXT_LENGTH).alias(f"{column_name}_preview")
        for column_name in json_columns
    ]

    print(f"\n{name} JSON/string column preview:")
    df.select("fullVisitorId", "visitId", *preview_expressions).show(
        PREVIEW_ROW_COUNT,
        truncate=120,
    )


show_json_preview("Train", train_sample_df)
show_json_preview("Test", test_sample_df)


Train sample rows: 100
Test sample rows: 100

Train session-level preview:
+--------+-------------------+----------+-----------+--------------+---------------+--------------------+
|    date|      fullVisitorId|   visitId|visitNumber|visitStartTime|channelGrouping|socialEngagementType|
+--------+-------------------+----------+-----------+--------------+---------------+--------------------+
|20171016|3162355547410993243|1508198450|          1|    1508198450| Organic Search|Not Socially Engaged|
|20171016|8934116514970143966|1508176307|          6|    1508176307|       Referral|Not Socially Engaged|
|20171016|7992466427990357681|1508201613|          1|    1508201613|         Direct|Not Socially Engaged|
|20171016|9075655783635761930|1508169851|          1|    1508169851| Organic Search|Not Socially Engaged|
|20171016|6960673291025684308|1508190552|          1|    1508190552| Organic Search|Not Socially Engaged|
+--------+-------------------+----------+-----------+--------------+---------

### Buoc 4: Chon cot can giu o muc session-level

Viec can lam trong buoc 4:
1. Xac dinh cac cot dinh danh session va visitor.
2. Giu cac cot context don gian nhu ngay, kenh truy cap, social engagement.
3. Giu cac cot JSON quan trong de parse thanh feature o cac buoc sau.
4. Tam thoi loai cot `hits` vi cot nay rat nang va nam o muc hit-level, khong phai session-level truc tiep.
5. Tao ham select cot session-level de tai su dung cho sample va full data.

In [5]:
SESSION_ID_COLUMNS = [
    "fullVisitorId",
    "visitId",
    "visitNumber",
    "visitStartTime",
]

SESSION_TIME_COLUMNS = [
    "date",
]

SESSION_CONTEXT_COLUMNS = [
    "channelGrouping",
    "socialEngagementType",
]

SESSION_JSON_COLUMNS = [
    "totals",
    "device",
    "geoNetwork",
    "trafficSource",
    "customDimensions",
]

SESSION_LEVEL_COLUMNS = (
    SESSION_ID_COLUMNS
    + SESSION_TIME_COLUMNS
    + SESSION_CONTEXT_COLUMNS
    + SESSION_JSON_COLUMNS
)


def validate_columns_exist(df, required_columns):
    missing_columns = sorted(set(required_columns) - set(df.columns))
    if missing_columns:
        raise ValueError(f"Missing columns: {missing_columns}")


def select_session_level_columns(df):
    validate_columns_exist(df, SESSION_LEVEL_COLUMNS)
    return df.select(*SESSION_LEVEL_COLUMNS)


train_session_sample_df = select_session_level_columns(train_sample_df).cache()
test_session_sample_df = select_session_level_columns(test_sample_df).cache()

dropped_columns = sorted(set(train_sample_df.columns) - set(SESSION_LEVEL_COLUMNS))

print("Session-level columns to keep:")
for index, column_name in enumerate(SESSION_LEVEL_COLUMNS, start=1):
    print(f"{index:02d}. {column_name}")

print("\nDropped columns at this step:", dropped_columns)
print("Train session sample rows:", train_session_sample_df.count())
print("Test session sample rows:", test_session_sample_df.count())

print("\nTrain session-level selected schema:")
train_session_sample_df.printSchema()

print("\nTrain session-level selected preview:")
train_session_sample_df.select(
    "fullVisitorId",
    "visitId",
    "date",
    "channelGrouping",
    "totals",
).show(PREVIEW_ROW_COUNT, truncate=120)


Session-level columns to keep:
01. fullVisitorId
02. visitId
03. visitNumber
04. visitStartTime
05. date
06. channelGrouping
07. socialEngagementType
08. totals
09. device
10. geoNetwork
11. trafficSource
12. customDimensions

Dropped columns at this step: ['hits']
Train session sample rows: 100
Test session sample rows: 100

Train session-level selected schema:
root
 |-- fullVisitorId: string (nullable = true)
 |-- visitId: string (nullable = true)
 |-- visitNumber: string (nullable = true)
 |-- visitStartTime: string (nullable = true)
 |-- date: string (nullable = true)
 |-- channelGrouping: string (nullable = true)
 |-- socialEngagementType: string (nullable = true)
 |-- totals: string (nullable = true)
 |-- device: string (nullable = true)
 |-- geoNetwork: string (nullable = true)
 |-- trafficSource: string (nullable = true)
 |-- customDimensions: string (nullable = true)


Train session-level selected preview:
+-------------------+----------+--------+---------------+--------------

### Buoc 5: Parse cac cot JSON quan trong

Viec can lam trong buoc 5:
1. Khai bao schema cho cac cot JSON session-level: `totals`, `device`, `geoNetwork`, `trafficSource`.
2. Dung `from_json` de parse string JSON thanh struct.
3. Tao cac cot feature tam thoi co prefix ro rang: `totals_*`, `device_*`, `geo_*`, `traffic_*`.
4. Trich `customDimensions.value` bang regex vi cot nay dang o dang Python-like string, khong phai JSON chuan.
5. Giu lai cot JSON goc trong buoc nay; viec drop cot goc se lam o buoc 9.

In [6]:
from pyspark.sql.types import BooleanType, StringType, StructField, StructType


totals_schema = StructType([
    StructField("visits", StringType(), True),
    StructField("hits", StringType(), True),
    StructField("pageviews", StringType(), True),
    StructField("bounces", StringType(), True),
    StructField("newVisits", StringType(), True),
    StructField("timeOnSite", StringType(), True),
    StructField("sessionQualityDim", StringType(), True),
    StructField("transactions", StringType(), True),
    StructField("transactionRevenue", StringType(), True),
    StructField("totalTransactionRevenue", StringType(), True),
])

device_schema = StructType([
    StructField("browser", StringType(), True),
    StructField("operatingSystem", StringType(), True),
    StructField("isMobile", BooleanType(), True),
    StructField("deviceCategory", StringType(), True),
])

geo_network_schema = StructType([
    StructField("continent", StringType(), True),
    StructField("subContinent", StringType(), True),
    StructField("country", StringType(), True),
    StructField("region", StringType(), True),
    StructField("metro", StringType(), True),
    StructField("city", StringType(), True),
    StructField("networkDomain", StringType(), True),
])

adwords_click_info_schema = StructType([
    StructField("campaignId", StringType(), True),
    StructField("adGroupId", StringType(), True),
    StructField("creativeId", StringType(), True),
    StructField("criteriaId", StringType(), True),
    StructField("page", StringType(), True),
    StructField("slot", StringType(), True),
    StructField("gclId", StringType(), True),
    StructField("isVideoAd", BooleanType(), True),
])

traffic_source_schema = StructType([
    StructField("campaign", StringType(), True),
    StructField("source", StringType(), True),
    StructField("medium", StringType(), True),
    StructField("keyword", StringType(), True),
    StructField("referralPath", StringType(), True),
    StructField("adContent", StringType(), True),
    StructField("isTrueDirect", BooleanType(), True),
    StructField("adwordsClickInfo", adwords_click_info_schema, True),
])


def parse_session_json_columns(df):
    parsed_df = (
        df
        .withColumn("totals_struct", F.from_json(F.col("totals"), totals_schema))
        .withColumn("device_struct", F.from_json(F.col("device"), device_schema))
        .withColumn("geo_struct", F.from_json(F.col("geoNetwork"), geo_network_schema))
        .withColumn("traffic_struct", F.from_json(F.col("trafficSource"), traffic_source_schema))
    )

    return parsed_df.select(
        "*",
        F.col("totals_struct.visits").alias("totals_visits_raw"),
        F.col("totals_struct.hits").alias("totals_hits_raw"),
        F.col("totals_struct.pageviews").alias("totals_pageviews_raw"),
        F.col("totals_struct.bounces").alias("totals_bounces_raw"),
        F.col("totals_struct.newVisits").alias("totals_new_visits_raw"),
        F.col("totals_struct.timeOnSite").alias("totals_time_on_site_raw"),
        F.col("totals_struct.sessionQualityDim").alias("totals_session_quality_dim_raw"),
        F.col("totals_struct.transactions").alias("totals_transactions_raw"),
        F.col("totals_struct.transactionRevenue").alias("totals_transaction_revenue_raw"),
        F.col("totals_struct.totalTransactionRevenue").alias("totals_total_transaction_revenue_raw"),
        F.col("device_struct.browser").alias("device_browser"),
        F.col("device_struct.operatingSystem").alias("device_operating_system"),
        F.col("device_struct.isMobile").alias("device_is_mobile"),
        F.col("device_struct.deviceCategory").alias("device_category"),
        F.col("geo_struct.continent").alias("geo_continent"),
        F.col("geo_struct.subContinent").alias("geo_sub_continent"),
        F.col("geo_struct.country").alias("geo_country"),
        F.col("geo_struct.region").alias("geo_region"),
        F.col("geo_struct.metro").alias("geo_metro"),
        F.col("geo_struct.city").alias("geo_city"),
        F.col("geo_struct.networkDomain").alias("geo_network_domain"),
        F.col("traffic_struct.campaign").alias("traffic_campaign"),
        F.col("traffic_struct.source").alias("traffic_source"),
        F.col("traffic_struct.medium").alias("traffic_medium"),
        F.col("traffic_struct.keyword").alias("traffic_keyword"),
        F.col("traffic_struct.referralPath").alias("traffic_referral_path"),
        F.col("traffic_struct.adContent").alias("traffic_ad_content"),
        F.col("traffic_struct.isTrueDirect").alias("traffic_is_true_direct"),
        F.col("traffic_struct.adwordsClickInfo.gclId").alias("traffic_gcl_id"),
        F.regexp_extract(F.col("customDimensions"), r"'value': '([^']+)'", 1).alias("custom_dimension_value"),
    )


train_parsed_sample_df = parse_session_json_columns(train_session_sample_df).cache()
test_parsed_sample_df = parse_session_json_columns(test_session_sample_df).cache()

parsed_preview_columns = [
    "fullVisitorId",
    "visitId",
    "date",
    "totals_hits_raw",
    "totals_pageviews_raw",
    "totals_transaction_revenue_raw",
    "device_browser",
    "device_category",
    "geo_country",
    "traffic_source",
    "traffic_medium",
    "custom_dimension_value",
]

print("Parsed train sample rows:", train_parsed_sample_df.count())
print("Parsed test sample rows:", test_parsed_sample_df.count())

print("\nParsed columns added:")
for column_name in parsed_preview_columns[3:]:
    print("-", column_name)

print("\nParsed train sample preview:")
train_parsed_sample_df.select(*parsed_preview_columns).show(PREVIEW_ROW_COUNT, truncate=100)

print("\nParsed train sample schema preview:")
train_parsed_sample_df.select(*parsed_preview_columns).printSchema()


Parsed train sample rows: 100
Parsed test sample rows: 100

Parsed columns added:
- totals_hits_raw
- totals_pageviews_raw
- totals_transaction_revenue_raw
- device_browser
- device_category
- geo_country
- traffic_source
- traffic_medium
- custom_dimension_value

Parsed train sample preview:
+-------------------+----------+--------+---------------+--------------------+------------------------------+--------------+---------------+-------------+----------------+--------------+----------------------+
|      fullVisitorId|   visitId|    date|totals_hits_raw|totals_pageviews_raw|totals_transaction_revenue_raw|device_browser|device_category|  geo_country|  traffic_source|traffic_medium|custom_dimension_value|
+-------------------+----------+--------+---------------+--------------------+------------------------------+--------------+---------------+-------------+----------------+--------------+----------------------+
|3162355547410993243|1508198450|20171016|              1|                   

### Buoc 6: Tao cac cot numeric sach

Viec can lam trong buoc 6:
1. Chuyen cac cot id/thoi gian tu string sang kieu so hoac date/timestamp phu hop.
2. Chuyen cac chi so trong `totals_*_raw` sang integer/long.
3. Thay null bang 0 cho cac metric dang dem nhu hits, pageviews, transactions, revenue.
4. Tao revenue dang `long` theo micros va revenue dang `double` theo don vi tien te de de doc hon.
5. Kiem tra schema va preview de dam bao cac cot numeric da sach.

In [7]:
REVENUE_MICRO_DIVISOR = 1_000_000.0


def clean_int_column(column_name, default_value=0):
    return F.coalesce(F.col(column_name).cast("int"), F.lit(default_value))


def clean_long_column(column_name, default_value=0):
    return F.coalesce(F.col(column_name).cast("long"), F.lit(default_value))


def create_clean_numeric_columns(df):
    cleaned_df = (
        df
        .withColumn("visit_id", F.col("visitId").cast("long"))
        .withColumn("visit_number", F.col("visitNumber").cast("int"))
        .withColumn("visit_start_time", F.col("visitStartTime").cast("long"))
        .withColumn("visit_start_timestamp", F.to_timestamp(F.from_unixtime(F.col("visitStartTime").cast("long"))))
        .withColumn("session_date", F.to_date(F.col("date"), "yyyyMMdd"))
        .withColumn("totals_visits", clean_int_column("totals_visits_raw"))
        .withColumn("totals_hits", clean_int_column("totals_hits_raw"))
        .withColumn("totals_pageviews", clean_int_column("totals_pageviews_raw"))
        .withColumn("totals_bounces", clean_int_column("totals_bounces_raw"))
        .withColumn("totals_new_visits", clean_int_column("totals_new_visits_raw"))
        .withColumn("totals_time_on_site", clean_int_column("totals_time_on_site_raw"))
        .withColumn("totals_session_quality_dim", clean_int_column("totals_session_quality_dim_raw"))
        .withColumn("totals_transactions", clean_int_column("totals_transactions_raw"))
        .withColumn("totals_transaction_revenue", clean_long_column("totals_transaction_revenue_raw"))
        .withColumn("totals_total_transaction_revenue", clean_long_column("totals_total_transaction_revenue_raw"))
    )

    return (
        cleaned_df
        .withColumn("transaction_revenue", F.col("totals_transaction_revenue") / F.lit(REVENUE_MICRO_DIVISOR))
        .withColumn("total_transaction_revenue", F.col("totals_total_transaction_revenue") / F.lit(REVENUE_MICRO_DIVISOR))
    )


train_numeric_sample_df = create_clean_numeric_columns(train_parsed_sample_df).cache()
test_numeric_sample_df = create_clean_numeric_columns(test_parsed_sample_df).cache()

numeric_preview_columns = [
    "fullVisitorId",
    "visit_id",
    "visit_number",
    "session_date",
    "visit_start_timestamp",
    "totals_hits",
    "totals_pageviews",
    "totals_time_on_site",
    "totals_transactions",
    "totals_total_transaction_revenue",
    "total_transaction_revenue",
]

print("Train numeric sample rows:", train_numeric_sample_df.count())
print("Test numeric sample rows:", test_numeric_sample_df.count())

print("\nNumeric columns preview:")
train_numeric_sample_df.select(*numeric_preview_columns).show(PREVIEW_ROW_COUNT, truncate=100)

print("\nNumeric columns schema:")
train_numeric_sample_df.select(*numeric_preview_columns).printSchema()

print("\nQuick numeric summary:")
train_numeric_sample_df.select(
    "totals_hits",
    "totals_pageviews",
    "totals_time_on_site",
    "totals_transactions",
    "total_transaction_revenue",
).summary("count", "min", "mean", "max").show(truncate=False)


Train numeric sample rows: 100
Test numeric sample rows: 100

Numeric columns preview:
+-------------------+----------+------------+------------+---------------------+-----------+----------------+-------------------+-------------------+--------------------------------+-------------------------+
|      fullVisitorId|  visit_id|visit_number|session_date|visit_start_timestamp|totals_hits|totals_pageviews|totals_time_on_site|totals_transactions|totals_total_transaction_revenue|total_transaction_revenue|
+-------------------+----------+------------+------------+---------------------+-----------+----------------+-------------------+-------------------+--------------------------------+-------------------------+
|3162355547410993243|1508198450|           1|  2017-10-16|  2017-10-17 07:00:50|          1|               1|                  0|                  0|                               0|                      0.0|
|8934116514970143966|1508176307|           6|  2017-10-16|  2017-10-17 00:51:

### Buoc 7: Tao label mua hang

Viec can lam trong buoc 7:
1. Xac dinh rule tao label cho bai toan phan loai session co mua hang hay khong.
2. Tao label `made_purchase`: 1 neu session co transaction hoac revenue > 0, nguoc lai la 0.
3. Tao them cot `purchase_signal_reason` de debug label den tu transaction hay revenue.
4. Kiem tra phan bo label tren sample train/test.
5. Preview cac session mua hang neu sample co du lieu positive.

In [8]:
def add_purchase_label(df):
    has_transaction = F.col("totals_transactions") > 0
    has_transaction_revenue = F.col("totals_transaction_revenue") > 0
    has_total_transaction_revenue = F.col("totals_total_transaction_revenue") > 0
    has_purchase_signal = has_transaction | has_transaction_revenue | has_total_transaction_revenue

    return (
        df
        .withColumn("made_purchase", F.when(has_purchase_signal, F.lit(1)).otherwise(F.lit(0)))
        .withColumn(
            "purchase_signal_reason",
            F.when(has_total_transaction_revenue, F.lit("total_transaction_revenue"))
            .when(has_transaction_revenue, F.lit("transaction_revenue"))
            .when(has_transaction, F.lit("transactions"))
            .otherwise(F.lit("no_purchase_signal")),
        )
    )


train_labeled_sample_df = add_purchase_label(train_numeric_sample_df).cache()
test_labeled_sample_df = add_purchase_label(test_numeric_sample_df).cache()

label_preview_columns = [
    "fullVisitorId",
    "visit_id",
    "session_date",
    "totals_transactions",
    "totals_transaction_revenue",
    "totals_total_transaction_revenue",
    "transaction_revenue",
    "total_transaction_revenue",
    "made_purchase",
    "purchase_signal_reason",
]

print("Train labeled sample rows:", train_labeled_sample_df.count())
print("Test labeled sample rows:", test_labeled_sample_df.count())

print("\nTrain label distribution:")
train_labeled_sample_df.groupBy("made_purchase").count().orderBy("made_purchase").show()

print("\nTest label distribution:")
test_labeled_sample_df.groupBy("made_purchase").count().orderBy("made_purchase").show()

print("\nTrain label preview:")
train_labeled_sample_df.select(*label_preview_columns).show(PREVIEW_ROW_COUNT, truncate=100)

print("\nPositive purchase sessions in train sample:")
train_labeled_sample_df.filter(F.col("made_purchase") == 1).select(*label_preview_columns).show(
    PREVIEW_ROW_COUNT,
    truncate=100,
)


Train labeled sample rows: 100
Test labeled sample rows: 100

Train label distribution:
+-------------+-----+
|made_purchase|count|
+-------------+-----+
|            0|  100|
+-------------+-----+


Test label distribution:
+-------------+-----+
|made_purchase|count|
+-------------+-----+
|            0|  100|
+-------------+-----+


Train label preview:
+-------------------+----------+------------+-------------------+--------------------------+--------------------------------+-------------------+-------------------------+-------------+----------------------+
|      fullVisitorId|  visit_id|session_date|totals_transactions|totals_transaction_revenue|totals_total_transaction_revenue|transaction_revenue|total_transaction_revenue|made_purchase|purchase_signal_reason|
+-------------------+----------+------------+-------------------+--------------------------+--------------------------------+-------------------+-------------------------+-------------+----------------------+
|31623555474109

### Buoc 8: Parse them device, geoNetwork, trafficSource va cot quan trong

Viec can lam trong buoc 8:
1. Tao them feature tu ngay/thoi gian: nam, thang, ngay trong tuan, gio bat dau session.
2. Chuan hoa mot so cot traffic thanh nhom de EDA/model de dung hon.
3. Tao flag traffic quan trong: direct, paid, organic, referral, co keyword, co gclid.
4. Chuan hoa device va geo thanh nhom bo sung: browser family, OS family, region availability.
5. Tao DataFrame sample da enrich de dung cho cac buoc drop cot goc va ghi parquet sau nay.

In [9]:
def add_enriched_session_features(df):
    lower_medium = F.lower(F.coalesce(F.col("traffic_medium"), F.lit("")))
    lower_source = F.lower(F.coalesce(F.col("traffic_source"), F.lit("")))
    lower_browser = F.lower(F.coalesce(F.col("device_browser"), F.lit("")))
    lower_os = F.lower(F.coalesce(F.col("device_operating_system"), F.lit("")))

    return (
        df
        .withColumn("session_year", F.year("session_date"))
        .withColumn("session_month", F.month("session_date"))
        .withColumn("session_day_of_week", F.dayofweek("session_date"))
        .withColumn("session_hour", F.hour("visit_start_timestamp"))
        .withColumn("traffic_medium_clean", F.coalesce(F.col("traffic_medium"), F.lit("(not set)")))
        .withColumn("traffic_source_clean", F.coalesce(F.col("traffic_source"), F.lit("(not set)")))
        .withColumn(
            "traffic_channel_type",
            F.when(lower_medium == "organic", F.lit("organic_search"))
            .when(lower_medium.isin("cpc", "ppc", "paidsearch", "paid search"), F.lit("paid_search"))
            .when(lower_medium.isin("referral", "affiliate"), F.lit("referral"))
            .when(lower_medium == "email", F.lit("email"))
            .when(lower_medium.isin("social", "social-network", "social-media"), F.lit("social"))
            .when((lower_medium == "(none)") | (lower_source == "(direct)"), F.lit("direct"))
            .otherwise(F.lit("other")),
        )
        .withColumn("is_direct_traffic", (F.col("traffic_channel_type") == "direct").cast("int"))
        .withColumn("is_paid_traffic", (F.col("traffic_channel_type") == "paid_search").cast("int"))
        .withColumn("is_organic_traffic", (F.col("traffic_channel_type") == "organic_search").cast("int"))
        .withColumn("is_referral_traffic", (F.col("traffic_channel_type") == "referral").cast("int"))
        .withColumn(
            "has_traffic_keyword",
            (
                F.col("traffic_keyword").isNotNull()
                & ~F.lower(F.col("traffic_keyword")).isin("(not provided)", "(not set)")
            ).cast("int"),
        )
        .withColumn("has_gclid", F.col("traffic_gcl_id").isNotNull().cast("int"))
        .withColumn(
            "browser_family",
            F.when(lower_browser.contains("chrome"), F.lit("Chrome"))
            .when(lower_browser.contains("safari"), F.lit("Safari"))
            .when(lower_browser.contains("firefox"), F.lit("Firefox"))
            .when(lower_browser.contains("edge"), F.lit("Edge"))
            .when(lower_browser.contains("internet explorer"), F.lit("Internet Explorer"))
            .otherwise(F.lit("Other")),
        )
        .withColumn(
            "os_family",
            F.when(lower_os.contains("windows"), F.lit("Windows"))
            .when(lower_os.contains("macintosh") | lower_os.contains("mac os"), F.lit("Mac OS"))
            .when(lower_os.contains("android"), F.lit("Android"))
            .when(lower_os.contains("ios") | lower_os.contains("iphone") | lower_os.contains("ipad"), F.lit("iOS"))
            .when(lower_os.contains("linux"), F.lit("Linux"))
            .otherwise(F.lit("Other")),
        )
        .withColumn(
            "geo_region_clean",
            F.when(
                F.col("geo_region").isNull()
                | F.col("geo_region").isin("not available in demo dataset", "(not set)"),
                F.lit(None),
            ).otherwise(F.col("geo_region")),
        )
        .withColumn("has_geo_region", F.col("geo_region_clean").isNotNull().cast("int"))
    )


train_enriched_sample_df = add_enriched_session_features(train_labeled_sample_df).cache()
test_enriched_sample_df = add_enriched_session_features(test_labeled_sample_df).cache()

enriched_preview_columns = [
    "fullVisitorId",
    "visit_id",
    "session_date",
    "session_month",
    "session_day_of_week",
    "session_hour",
    "device_category",
    "browser_family",
    "os_family",
    "geo_country",
    "has_geo_region",
    "traffic_channel_type",
    "is_direct_traffic",
    "is_paid_traffic",
    "is_organic_traffic",
    "has_traffic_keyword",
    "has_gclid",
    "made_purchase",
]

print("Train enriched sample rows:", train_enriched_sample_df.count())
print("Test enriched sample rows:", test_enriched_sample_df.count())

print("\nEnriched feature preview:")
train_enriched_sample_df.select(*enriched_preview_columns).show(PREVIEW_ROW_COUNT, truncate=100)

print("\nTraffic channel distribution in train sample:")
train_enriched_sample_df.groupBy("traffic_channel_type").count().orderBy(F.desc("count")).show(truncate=False)

print("\nDevice category distribution in train sample:")
train_enriched_sample_df.groupBy("device_category", "browser_family", "os_family").count().orderBy(F.desc("count")).show(
    PREVIEW_ROW_COUNT,
    truncate=False,
)


Train enriched sample rows: 100
Test enriched sample rows: 100

Enriched feature preview:
+-------------------+----------+------------+-------------+-------------------+------------+---------------+--------------+---------+-------------+--------------+--------------------+-----------------+---------------+------------------+-------------------+---------+-------------+
|      fullVisitorId|  visit_id|session_date|session_month|session_day_of_week|session_hour|device_category|browser_family|os_family|  geo_country|has_geo_region|traffic_channel_type|is_direct_traffic|is_paid_traffic|is_organic_traffic|has_traffic_keyword|has_gclid|made_purchase|
+-------------------+----------+------------+-------------+-------------------+------------+---------------+--------------+---------+-------------+--------------+--------------------+-----------------+---------------+------------------+-------------------+---------+-------------+
|3162355547410993243|1508198450|  2017-10-16|           10|        

### Buoc 9: Drop cac cot JSON goc nang

Viec can lam trong buoc 9:
1. Xac dinh cac cot JSON goc can bo: `totals`, `device`, `geoNetwork`, `trafficSource`, `customDimensions`.
2. Drop cac cot struct tam thoi tao ra trong qua trinh parse JSON.
3. Drop cac cot raw trung gian neu da co cot numeric/categorical sach thay the.
4. Tao DataFrame sample sach hon: `train_clean_sample_df`, `test_clean_sample_df`.
5. Kiem tra lai schema va so cot truoc/sau khi drop.

In [10]:
ORIGINAL_JSON_COLUMNS_TO_DROP = [
    "totals",
    "device",
    "geoNetwork",
    "trafficSource",
    "customDimensions",
]

TEMP_STRUCT_COLUMNS_TO_DROP = [
    "totals_struct",
    "device_struct",
    "geo_struct",
    "traffic_struct",
]

RAW_PARSED_COLUMNS_TO_DROP = [
    "totals_visits_raw",
    "totals_hits_raw",
    "totals_pageviews_raw",
    "totals_bounces_raw",
    "totals_new_visits_raw",
    "totals_time_on_site_raw",
    "totals_session_quality_dim_raw",
    "totals_transactions_raw",
    "totals_transaction_revenue_raw",
    "totals_total_transaction_revenue_raw",
]

COLUMNS_TO_DROP_AFTER_PARSE = (
    ORIGINAL_JSON_COLUMNS_TO_DROP
    + TEMP_STRUCT_COLUMNS_TO_DROP
    + RAW_PARSED_COLUMNS_TO_DROP
)


def drop_heavy_json_columns(df):
    columns_present = [column_name for column_name in COLUMNS_TO_DROP_AFTER_PARSE if column_name in df.columns]
    return df.drop(*columns_present), columns_present


train_clean_sample_df, train_dropped_columns = drop_heavy_json_columns(train_enriched_sample_df)
test_clean_sample_df, test_dropped_columns = drop_heavy_json_columns(test_enriched_sample_df)

train_clean_sample_df = train_clean_sample_df.cache()
test_clean_sample_df = test_clean_sample_df.cache()

print("Train columns before drop:", len(train_enriched_sample_df.columns))
print("Train columns after drop:", len(train_clean_sample_df.columns))
print("Test columns before drop:", len(test_enriched_sample_df.columns))
print("Test columns after drop:", len(test_clean_sample_df.columns))

print("\nDropped columns from train:")
for column_name in train_dropped_columns:
    print("-", column_name)

remaining_heavy_columns = sorted(set(ORIGINAL_JSON_COLUMNS_TO_DROP + TEMP_STRUCT_COLUMNS_TO_DROP) & set(train_clean_sample_df.columns))
print("\nRemaining heavy JSON/temp columns:", remaining_heavy_columns)

clean_preview_columns = [
    "fullVisitorId",
    "visit_id",
    "session_date",
    "channelGrouping",
    "device_category",
    "browser_family",
    "geo_country",
    "traffic_channel_type",
    "totals_hits",
    "totals_pageviews",
    "total_transaction_revenue",
    "made_purchase",
]

print("\nClean sample preview:")
train_clean_sample_df.select(*clean_preview_columns).show(PREVIEW_ROW_COUNT, truncate=100)

print("\nClean sample schema preview:")
train_clean_sample_df.select(*clean_preview_columns).printSchema()


Train columns before drop: 82
Train columns after drop: 63
Test columns before drop: 82
Test columns after drop: 63

Dropped columns from train:
- totals
- device
- geoNetwork
- trafficSource
- customDimensions
- totals_struct
- device_struct
- geo_struct
- traffic_struct
- totals_visits_raw
- totals_hits_raw
- totals_pageviews_raw
- totals_bounces_raw
- totals_new_visits_raw
- totals_time_on_site_raw
- totals_session_quality_dim_raw
- totals_transactions_raw
- totals_transaction_revenue_raw
- totals_total_transaction_revenue_raw

Remaining heavy JSON/temp columns: []

Clean sample preview:
+-------------------+----------+------------+---------------+---------------+--------------+-------------+--------------------+-----------+----------------+-------------------------+-------------+
|      fullVisitorId|  visit_id|session_date|channelGrouping|device_category|browser_family|  geo_country|traffic_channel_type|totals_hits|totals_pageviews|total_transaction_revenue|made_purchase|
+-------

### Buoc 10: Thu sample ra Parquet

Viec can lam trong buoc 10:
1. Ghi `train_clean_sample_df` va `test_clean_sample_df` ra Parquet sample truoc khi ghi full data.
2. Dung `overwrite` cho sample de co the chay lai nhieu lan trong luc hoc/thuc nghiem.
3. Doc lai file Parquet sample vua ghi de kiem tra count va schema.
4. So sanh so cot truoc/sau khi doc lai Parquet.
5. Xac nhan duong dan sample Parquet da san sang cho cac buoc tiep theo.

In [11]:
SAMPLE_TRAIN_PARQUET_PATH = PARQUET_DIR / "sample_train_sessions"
SAMPLE_TEST_PARQUET_PATH = PARQUET_DIR / "sample_test_sessions"


def write_parquet_sample(df, output_path):
    (
        df
        .coalesce(1)
        .write
        .mode("overwrite")
        .parquet(str(output_path))
    )


PARQUET_DIR.mkdir(parents=True, exist_ok=True)

write_parquet_sample(train_clean_sample_df, SAMPLE_TRAIN_PARQUET_PATH)
write_parquet_sample(test_clean_sample_df, SAMPLE_TEST_PARQUET_PATH)

train_sample_parquet_df = spark.read.parquet(str(SAMPLE_TRAIN_PARQUET_PATH))
test_sample_parquet_df = spark.read.parquet(str(SAMPLE_TEST_PARQUET_PATH))

print("Sample train parquet path:", SAMPLE_TRAIN_PARQUET_PATH)
print("Sample test parquet path:", SAMPLE_TEST_PARQUET_PATH)

print("\nTrain clean sample rows before write:", train_clean_sample_df.count())
print("Train sample parquet rows after read:", train_sample_parquet_df.count())
print("Train clean sample columns before write:", len(train_clean_sample_df.columns))
print("Train sample parquet columns after read:", len(train_sample_parquet_df.columns))

print("\nTest clean sample rows before write:", test_clean_sample_df.count())
print("Test sample parquet rows after read:", test_sample_parquet_df.count())
print("Test clean sample columns before write:", len(test_clean_sample_df.columns))
print("Test sample parquet columns after read:", len(test_sample_parquet_df.columns))

print("\nRead-back train sample schema:")
train_sample_parquet_df.printSchema()

print("\nRead-back train sample preview:")
train_sample_parquet_df.select(*clean_preview_columns).show(PREVIEW_ROW_COUNT, truncate=100)


Sample train parquet path: g:\ds\data_pyspark_parquet\sample_train_sessions
Sample test parquet path: g:\ds\data_pyspark_parquet\sample_test_sessions

Train clean sample rows before write: 100
Train sample parquet rows after read: 100
Train clean sample columns before write: 63
Train sample parquet columns after read: 63

Test clean sample rows before write: 100
Test sample parquet rows after read: 100
Test clean sample columns before write: 63
Test sample parquet columns after read: 63

Read-back train sample schema:
root
 |-- fullVisitorId: string (nullable = true)
 |-- visitId: string (nullable = true)
 |-- visitNumber: string (nullable = true)
 |-- visitStartTime: string (nullable = true)
 |-- date: string (nullable = true)
 |-- channelGrouping: string (nullable = true)
 |-- socialEngagementType: string (nullable = true)
 |-- device_browser: string (nullable = true)
 |-- device_operating_system: string (nullable = true)
 |-- device_is_mobile: boolean (nullable = true)
 |-- device_c

### Buoc 11: Ghi full train ra Parquet

Viec can lam trong buoc 11:
1. Doc full `train_v2.csv` bang Spark, khong dung sample nua.
2. Ap dung lai toan bo pipeline da kiem tra tren sample: select cot, parse JSON, tao numeric, tao label, enrich feature, drop cot nang.
3. Ghi output ra `data_pyspark_parquet/train_sessions/`.
4. Partition output theo `session_year`, `session_month` de cac buoc EDA sau doc nhanh hon.
5. In thong tin duong dan, so cot va schema sau cung de kiem tra truoc khi chuyen sang test.

In [12]:
FULL_TRAIN_WRITE_MODE = "overwrite"
FULL_TRAIN_REPARTITION_COUNT = None  # None = khong ep shuffle repartition khi ghi full data.
FULL_TRAIN_MAX_RECORDS_PER_FILE = 500_000


def build_clean_session_df(csv_path):
    raw_df = spark.read.options(**CSV_READ_OPTIONS).csv(str(csv_path))
    session_df = select_session_level_columns(raw_df)
    parsed_df = parse_session_json_columns(session_df)
    numeric_df = create_clean_numeric_columns(parsed_df)
    labeled_df = add_purchase_label(numeric_df)
    enriched_df = add_enriched_session_features(labeled_df)
    clean_df, dropped_columns = drop_heavy_json_columns(enriched_df)
    return clean_df, dropped_columns


train_clean_full_df, full_train_dropped_columns = build_clean_session_df(TRAIN_CSV_PATH)

print("Full train input:", TRAIN_CSV_PATH)
print("Full train output:", TRAIN_PARQUET_PATH)
print("Full train columns:", len(train_clean_full_df.columns))
print("Dropped columns:", full_train_dropped_columns)

print("\nFull train final schema:")
train_clean_full_df.printSchema()

train_writer_df = train_clean_full_df
if FULL_TRAIN_REPARTITION_COUNT is not None:
    train_writer_df = train_writer_df.repartition(
        FULL_TRAIN_REPARTITION_COUNT,
        "session_year",
        "session_month",
    )

(
    train_writer_df
    .write
    .mode(FULL_TRAIN_WRITE_MODE)
    .option("maxRecordsPerFile", FULL_TRAIN_MAX_RECORDS_PER_FILE)
    .partitionBy("session_year", "session_month")
    .parquet(str(TRAIN_PARQUET_PATH))
)

print("Done writing full train parquet:", TRAIN_PARQUET_PATH)


Full train input: g:\ds\data_pyspark\train_v2.csv
Full train output: g:\ds\data_pyspark_parquet\train_sessions
Full train columns: 63
Dropped columns: ['totals', 'device', 'geoNetwork', 'trafficSource', 'customDimensions', 'totals_struct', 'device_struct', 'geo_struct', 'traffic_struct', 'totals_visits_raw', 'totals_hits_raw', 'totals_pageviews_raw', 'totals_bounces_raw', 'totals_new_visits_raw', 'totals_time_on_site_raw', 'totals_session_quality_dim_raw', 'totals_transactions_raw', 'totals_transaction_revenue_raw', 'totals_total_transaction_revenue_raw']

Full train final schema:
root
 |-- fullVisitorId: string (nullable = true)
 |-- visitId: string (nullable = true)
 |-- visitNumber: string (nullable = true)
 |-- visitStartTime: string (nullable = true)
 |-- date: string (nullable = true)
 |-- channelGrouping: string (nullable = true)
 |-- socialEngagementType: string (nullable = true)
 |-- device_browser: string (nullable = true)
 |-- device_operating_system: string (nullable = true

### Buoc 12: Lam tuong tu voi test

Viec can lam trong buoc 12:
1. Doc full `test_v2.csv` bang Spark.
2. Ap dung lai pipeline clean session da dung cho train.
3. Ghi output ra `data_pyspark_parquet/test_sessions/`.
4. Partition theo `session_year`, `session_month` giong train de EDA/doc lai thong nhat.
5. In thong tin output va schema de kiem tra nhanh.

In [13]:
FULL_TEST_WRITE_MODE = "overwrite"
FULL_TEST_REPARTITION_COUNT = None
FULL_TEST_MAX_RECORDS_PER_FILE = 500_000


test_clean_full_df, full_test_dropped_columns = build_clean_session_df(TEST_CSV_PATH)

print("Full test input:", TEST_CSV_PATH)
print("Full test output:", TEST_PARQUET_PATH)
print("Full test columns:", len(test_clean_full_df.columns))
print("Dropped columns:", full_test_dropped_columns)

print("\nFull test final schema:")
test_clean_full_df.printSchema()

test_writer_df = test_clean_full_df
if FULL_TEST_REPARTITION_COUNT is not None:
    test_writer_df = test_writer_df.repartition(
        FULL_TEST_REPARTITION_COUNT,
        "session_year",
        "session_month",
    )

(
    test_writer_df
    .write
    .mode(FULL_TEST_WRITE_MODE)
    .option("maxRecordsPerFile", FULL_TEST_MAX_RECORDS_PER_FILE)
    .partitionBy("session_year", "session_month")
    .parquet(str(TEST_PARQUET_PATH))
)

print("Done writing full test parquet:", TEST_PARQUET_PATH)


Full test input: g:\ds\data_pyspark\test_v2.csv
Full test output: g:\ds\data_pyspark_parquet\test_sessions
Full test columns: 63
Dropped columns: ['totals', 'device', 'geoNetwork', 'trafficSource', 'customDimensions', 'totals_struct', 'device_struct', 'geo_struct', 'traffic_struct', 'totals_visits_raw', 'totals_hits_raw', 'totals_pageviews_raw', 'totals_bounces_raw', 'totals_new_visits_raw', 'totals_time_on_site_raw', 'totals_session_quality_dim_raw', 'totals_transactions_raw', 'totals_transaction_revenue_raw', 'totals_total_transaction_revenue_raw']

Full test final schema:
root
 |-- fullVisitorId: string (nullable = true)
 |-- visitId: string (nullable = true)
 |-- visitNumber: string (nullable = true)
 |-- visitStartTime: string (nullable = true)
 |-- date: string (nullable = true)
 |-- channelGrouping: string (nullable = true)
 |-- socialEngagementType: string (nullable = true)
 |-- device_browser: string (nullable = true)
 |-- device_operating_system: string (nullable = true)
 |--

### Buoc 13: Kiem tra lai sau khi ghi Parquet

Viec can lam trong buoc 13:
1. Kiem tra file `_SUCCESS` cua train/test Parquet.
2. Doc lai `train_sessions` va `test_sessions` tu Parquet.
3. Kiem tra so dong, so cot va schema sau khi doc lai.
4. Kiem tra partition `session_year`, `session_month`.
5. Kiem tra nhanh label va mot vai cot quan trong de dam bao du lieu da san sang cho EDA/model.

In [14]:
def parquet_success_exists(parquet_path):
    return (parquet_path / "_SUCCESS").exists()


def inspect_parquet_dataset(name, parquet_path):
    print(f"\n===== {name} =====")
    print("Path:", parquet_path)
    print("_SUCCESS exists:", parquet_success_exists(parquet_path))

    df = spark.read.parquet(str(parquet_path))

    print("Rows:", df.count())
    print("Columns:", len(df.columns))

    print("\nSchema:")
    df.printSchema()

    print("\nPartition distribution:")
    df.groupBy("session_year", "session_month").count().orderBy("session_year", "session_month").show(
        100,
        truncate=False,
    )

    print("\nLabel distribution:")
    df.groupBy("made_purchase").count().orderBy("made_purchase").show(truncate=False)

    print("\nImportant columns preview:")
    df.select(*clean_preview_columns).show(PREVIEW_ROW_COUNT, truncate=100)

    return df


train_parquet_df = inspect_parquet_dataset("train_sessions", TRAIN_PARQUET_PATH)
test_parquet_df = inspect_parquet_dataset("test_sessions", TEST_PARQUET_PATH)

train_columns = set(train_parquet_df.columns)
test_columns = set(test_parquet_df.columns)

print("\n===== Train/Test column check =====")
print("Columns only in train:", sorted(train_columns - test_columns))
print("Columns only in test:", sorted(test_columns - train_columns))
print("Same column count:", len(train_columns), len(test_columns))



===== train_sessions =====
Path: g:\ds\data_pyspark_parquet\train_sessions
_SUCCESS exists: True
Rows: 1708337
Columns: 63

Schema:
root
 |-- fullVisitorId: string (nullable = true)
 |-- visitId: string (nullable = true)
 |-- visitNumber: string (nullable = true)
 |-- visitStartTime: string (nullable = true)
 |-- date: string (nullable = true)
 |-- channelGrouping: string (nullable = true)
 |-- socialEngagementType: string (nullable = true)
 |-- device_browser: string (nullable = true)
 |-- device_operating_system: string (nullable = true)
 |-- device_is_mobile: boolean (nullable = true)
 |-- device_category: string (nullable = true)
 |-- geo_continent: string (nullable = true)
 |-- geo_sub_continent: string (nullable = true)
 |-- geo_country: string (nullable = true)
 |-- geo_region: string (nullable = true)
 |-- geo_metro: string (nullable = true)
 |-- geo_city: string (nullable = true)
 |-- geo_network_domain: string (nullable = true)
 |-- traffic_campaign: string (nullable = true)

### Buoc 14: Ghi lai ket qua log

Viec can lam trong buoc 14:
1. Tong hop metadata cua pipeline doc CSV sang Parquet.
2. Ghi lai input path, output path va trang thai `_SUCCESS`.
3. Ghi lai so dong, so cot, partition va phan bo label cua train/test.
4. Luu log ra `data_pyspark_parquet/read_csv_to_parquet_log.json`.
5. Doc lai log de kiem tra file log da ghi thanh cong.

In [15]:
import json
from datetime import datetime, timezone


def collect_partition_counts(df):
    return [
        row.asDict()
        for row in (
            df.groupBy("session_year", "session_month")
            .count()
            .orderBy("session_year", "session_month")
            .collect()
        )
    ]


def collect_label_counts(df):
    return [
        row.asDict()
        for row in df.groupBy("made_purchase").count().orderBy("made_purchase").collect()
    ]


def build_dataset_log(name, input_path, output_path, df):
    return {
        "name": name,
        "input_path": str(input_path),
        "output_path": str(output_path),
        "success_marker_exists": parquet_success_exists(output_path),
        "row_count": df.count(),
        "column_count": len(df.columns),
        "columns": df.columns,
        "partition_columns": ["session_year", "session_month"],
        "partition_counts": collect_partition_counts(df),
        "label_counts": collect_label_counts(df),
    }


pipeline_log = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "spark_version": spark.version,
    "spark_master": spark.sparkContext.master,
    "project_root": str(PROJECT_ROOT),
    "csv_read_options": CSV_READ_OPTIONS,
    "dropped_columns_after_parse": COLUMNS_TO_DROP_AFTER_PARSE,
    "datasets": [
        build_dataset_log("train_sessions", TRAIN_CSV_PATH, TRAIN_PARQUET_PATH, train_parquet_df),
        build_dataset_log("test_sessions", TEST_CSV_PATH, TEST_PARQUET_PATH, test_parquet_df),
    ],
}

PARQUET_DIR.mkdir(parents=True, exist_ok=True)
with LOG_PATH.open("w", encoding="utf-8") as log_file:
    json.dump(pipeline_log, log_file, ensure_ascii=False, indent=2, default=str)

print("Log path:", LOG_PATH)
print("Log file exists:", LOG_PATH.exists())

with LOG_PATH.open(encoding="utf-8") as log_file:
    saved_log = json.load(log_file)

print("Datasets logged:", [dataset["name"] for dataset in saved_log["datasets"]])
for dataset in saved_log["datasets"]:
    print(
        dataset["name"],
        "rows=", dataset["row_count"],
        "columns=", dataset["column_count"],
        "_SUCCESS=", dataset["success_marker_exists"],
    )


Log path: g:\ds\data_pyspark_parquet\read_csv_to_parquet_log.json
Log file exists: True
Datasets logged: ['train_sessions', 'test_sessions']
train_sessions rows= 1708337 columns= 63 _SUCCESS= True
test_sessions rows= 401589 columns= 63 _SUCCESS= True


# Tổng kết

Từ dữ liệu đầu vào là file csv rất nặng:
- data_pyspark/train_v2.csv
- data_pyspark/test_v2.csv

Ta đã tạo thành công các file parquet:
- data_pyspark_parquet/train_sessions
- data_pyspark_parquet/test_sessions
- data_pyspark_parquet/sample_train_sessions
- data_pyspark_parquet/sample_test_sessions
- data_pyspark_parquet/read_csv_to_parquet_log.json

Những việc đã làm:
1. Khởi tạo SparkSession local
2. Kiểm tra môi trường Java/PySpark
3. Đọc thử header/schema CSV
4. Đọc thử sample nhỏ từ train/test
5. Chọn cột session-level
6. Parse các cột JSON quan trọng: `totals`, `device`, `geoNetwork`, ...
7. Tạo numeric columns sạch: `hits`, `pageviews`, `revenue`, ...
8. Tạo label: `made_purchase` và `purchase_signal_reason` để phục vụ bài toán phân loại sau này
9. Enrich thêm feature: Các feature thời gian như `year`, `month`, `day`, `hour`; `traffic lags`, ...
10. Drop các cột JSON gốc nặng và cột raw trung gian
11. Ghi ra sample Parquet
12. Ghi ra full train và test.
13. Đọc lại để kiểm tra và ghi ra log JSON


Kết quả thàng công đã có:
- Tập train gồm : 
    - Số dòng 1.708.337 dòng
    - Số cột 63
    - partition theo 
        - session_year
        - session_month
    - Label train
- Tập test gồm :
    - Số dòng 401.589
    - Số cột 63
    - partition theo
        - session_year
        - session_month
    - Label test.
- Log đã tạo thành công.


Tóm lại: notebook này đã hoàn thành đúng mục tiêu ban đầu: đọc CSV lớn bằng Spark, parse/clean session-level, tạo feature/label, ghi ra Parquet và kiểm tra lại output thành công.